<a href="https://colab.research.google.com/github/lanehale/pytorch-deep-learning/blob/main/pytorch06_comparisons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Moved pip installs outside of py scripts as apparently it's best practice.
"""
# See if torchmetrics exists, if not, install it
try:
  import torchmetrics, mlxtend
  print("torchmetrics already installed.")
except:
  print("Installing torchmetrics...")
  !pip install -q torchmetrics -U mlxtend
  import torchmetrics, mlxtend
  print("Done installing torchmetrics.")

Installing torchmetrics...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.5/961.5 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.5 MB/s eta 0:00:00
Done installing torchmetrics.


In [2]:
"""
Run several transfer learning model experiments with different hyperparameters.
"""
"""
Create the going_modular folder and move in its scripts.
"""
import os

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
  from going_modular import data_setup, engine, pretrained, get_any_data
  print("going_modular scripts already downloaded.")
except:
  """
  This block attempts to download a GitHub repository,
  move a specific directory from the downloaded repository to the current working directory,
  and then remove the downloaded repository.
  """
  # Get the going_modular scripts
  print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")

  # Clone the git repository
  !git clone https://github.com/lanehale/pytorch-deep-learning

  # When cloning a GitHub repository, the directory structure on your local machine doesn't include /tree/main/, so it shouldn't be included in the mv command.
  # The . at the end of the command tells mv to move the specified directory into the current working directory.
  !mv pytorch-deep-learning/going_modular .

  # remove the downloaded repository
  !rm -rf pytorch-deep-learning

  # move these two files out to parent directory
  !mv going_modular/train.py .
  !mv going_modular/predict.py .

  from going_modular import data_setup, engine, pretrained, get_any_data

print(">!ls")
!ls
print(">!ls going_modular")
!ls going_modular

[INFO] Couldn't find going_modular scripts... downloading them from GitHub.
Cloning into 'pytorch-deep-learning'...
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 251 (delta 78), reused 13 (delta 13), pack-reused 129 (from 1)
Receiving objects: 100% (251/251), 5.78 MiB | 14.83 MiB/s, done.
Resolving deltas: 100% (133/133), done.
>!ls
going_modular  predict.py  runs  sample_data  train.py
>!ls going_modular
data_setup.py  get_any_data.py	   get_data.py	     pretrained.py  utils.py
engine.py      get_custom_data.py  model_builder.py  __pycache__


In [ ]:
# Get 10% dataset
get_any_data.from_path(from_path="pizza_steak_sushi.zip", image_dir="pizza_steak_sushi")

In [ ]:
import torch
import torchvision

from pathlib import Path
from going_modular import pretrained

# Setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

image_path = Path("data/pizza_steak_sushi")

# Set up dirs
train_dir = image_path / "train"
test_dir = image_path / "test"

test_image_path_list = list(Path(test_dir).glob("*/*.jpg"))

BATCH_SIZE = 32
dropout = 0.2
in_features = 1280
optimizer_type = "Adam"
optimizer_lr = 0.001

weights_b0 = torchvision.models.EfficientNet_B0_Weights.DEFAULT  # .DEFAULT = best available weights from pretraining on ImageNet
weights_b2 = torchvision.models.EfficientNet_B2_Weights.DEFAULT

"""
Train 10% dataset for 5 epochs using Eff_B0
"""
NUM_EPOCHS = 5

# Set up the model with pretrained weights and send it to the target device (torchvision v0.13+)
model_b0_5x_10p = torchvision.models.efficientnet_b0(weights=weights_b0).to(device)

results_b0_5x_10p, pred_list_b0_5x_10p = pretrained.run_model(
    model=model_b0_5x_10p,
    weights=weights_b0,
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=BATCH_SIZE,
    dropout=dropout,
    in_features=in_features,
    optimizer_type=optimizer_type,
    optimizer_lr=optimizer_lr,
    num_epochs=NUM_EPOCHS,
    image_data=test_image_path_list,
    device=device
)

In [ ]:
"""
Train 10% dataset for 10 epochs using Eff_B0
"""
NUM_EPOCHS = 10

# Set up the model with pretrained weights and send it to the target device (torchvision v0.13+)
model_b0_10x_10p = torchvision.models.efficientnet_b0(weights=weights_b0).to(device)

results_b0_10x_10p, pred_list_b0_10x_10p = pretrained.run_model(
    model=model_b0_10x_10p,
    weights=weights_b0,
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=BATCH_SIZE,
    dropout=dropout,
    in_features=in_features,
    optimizer_type=optimizer_type,
    optimizer_lr=optimizer_lr,
    num_epochs=NUM_EPOCHS,
    image_data=test_image_path_list,
    device=device
)

In [ ]:
"""
Train 10% dataset for 5 epochs using Eff_B2
"""
dropout = 0.3
in_features = 1408
NUM_EPOCHS = 5

# Set up the model with pretrained weights and send it to the target device (torchvision v0.13+)
model_b2_5x_10p = torchvision.models.efficientnet_b2(weights=weights_b2).to(device)

results_b2_5x_10p, pred_list_b2_5x_10p = pretrained.run_model(
    model=model_b2_5x_10p,
    weights=weights_b2,
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=BATCH_SIZE,
    dropout=dropout,
    in_features=in_features,
    optimizer_type=optimizer_type,
    optimizer_lr=optimizer_lr,
    num_epochs=NUM_EPOCHS,
    image_data=test_image_path_list,
    device=device
)

In [ ]:
"""
Train 10% dataset for 10 epochs using Eff_B2
"""
NUM_EPOCHS = 10

# Set up the model with pretrained weights and send it to the target device (torchvision v0.13+)
model_b2_10x_10p = torchvision.models.efficientnet_b2(weights=weights_b2).to(device)

results_b2_10x_10p, pred_list_b2_10x_10p = pretrained.run_model(
    model=model_b2_10x_10p,
    weights=weights_b2,
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=BATCH_SIZE,
    dropout=dropout,
    in_features=in_features,
    optimizer_type=optimizer_type,
    optimizer_lr=optimizer_lr,
    num_epochs=NUM_EPOCHS,
    image_data=test_image_path_list,
    device=device
)

In [ ]:
!rm -rf data/
!ls

# Get 20% dataset
from going_modular import get_any_data

get_any_data.from_path(from_path="pizza_steak_sushi_20_percent.zip", image_dir="pizza_steak_sushi")

In [ ]:
test_image_path_list_20 = list(Path(test_dir).glob("*/*.jpg"))

"""
Train 20% dataset for 5 epochs using Eff_B0
"""
dropout = 0.2
in_features = 1280
NUM_EPOCHS = 5

# Set up the model with pretrained weights and send it to the target device (torchvision v0.13+)
model_b0_5x_20p = torchvision.models.efficientnet_b0(weights=weights_b0).to(device)

results_b0_5x_20p, pred_list_b0_5x_20p = pretrained.run_model(
    model=model_b0_5x_20p,
    weights=weights_b0,
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=BATCH_SIZE,
    dropout=dropout,
    in_features=in_features,
    optimizer_type=optimizer_type,
    optimizer_lr=optimizer_lr,
    num_epochs=NUM_EPOCHS,
    image_data=test_image_path_list_20,
    device=device
)

In [ ]:
"""
Train 20% dataset for 10 epochs using Eff_B0
"""
NUM_EPOCHS = 10

# Set up the model with pretrained weights and send it to the target device (torchvision v0.13+)
model_b0_10x_20p = torchvision.models.efficientnet_b0(weights=weights_b0).to(device)

results_b0_10x_20p, pred_list_b0_10x_20p = pretrained.run_model(
    model=model_b0_10x_20p,
    weights=weights_b0,
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=BATCH_SIZE,
    dropout=dropout,
    in_features=in_features,
    optimizer_type=optimizer_type,
    optimizer_lr=optimizer_lr,
    num_epochs=NUM_EPOCHS,
    image_data=test_image_path_list_20,
    device=device
)

In [ ]:
"""
Train 20% dataset for 5 epochs using Eff_B2
"""
dropout = 0.3
in_features = 1408
NUM_EPOCHS = 5

# Set up the model with pretrained weights and send it to the target device (torchvision v0.13+)
model_b2_5x_20p = torchvision.models.efficientnet_b2(weights=weights_b2).to(device)

results_b2_5x_20p, pred_list_b2_5x_20p = pretrained.run_model(
    model=model_b2_5x_20p,
    weights=weights_b2,
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=BATCH_SIZE,
    dropout=dropout,
    in_features=in_features,
    optimizer_type=optimizer_type,
    optimizer_lr=optimizer_lr,
    num_epochs=NUM_EPOCHS,
    image_data=test_image_path_list_20,
    device=device
)

In [ ]:
"""
Train 20% dataset for 10 epochs using Eff_B2
"""
NUM_EPOCHS = 10

# Set up the model with pretrained weights and send it to the target device (torchvision v0.13+)
model_b2_10x_20p = torchvision.models.efficientnet_b2(weights=weights_b2).to(device)

results_b2_10x_20p, pred_list_b2_10x_20p = pretrained.run_model(
    model=model_b2_10x_20p,
    weights=weights_b2,
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=BATCH_SIZE,
    dropout=dropout,
    in_features=in_features,
    optimizer_type=optimizer_type,
    optimizer_lr=optimizer_lr,
    num_epochs=NUM_EPOCHS,
    image_data=test_image_path_list_20,
    device=device
)

In [ ]:
# Create a function to display results
def compare_results(pred_list, name):
  false_count = 0
  for pred in pred_list:
    if pred['correct'] == False:
      false_count += 1
  false_percent = 100 * false_count / len(pred_list)
  print(
      f"{name :<10} | False predictions: {false_count :<2} out of {len(pred_list) :<3}, "
      f"or {false_percent:5.2f}% wrong, "
      f"{(100.0 - false_percent):.2f}% right"
  )

In [ ]:
compare_results(pred_list_b0_5x_10p, "b0_5x_10p")
compare_results(pred_list_b2_5x_10p, "b2_5x_10p")
compare_results(pred_list_b0_10x_10p, "b0_10x_10p")
compare_results(pred_list_b2_10x_10p, "b2_10x_10p")
compare_results(pred_list_b0_5x_20p, "b0_5x_20p")
compare_results(pred_list_b2_5x_20p, "b0_5x_20p")
compare_results(pred_list_b0_10x_20p, "b0_10x_20p")
compare_results(pred_list_b2_10x_20p, "b2_10x_20p")

In [ ]:
print(f"Max test acc: {max(results_b0_5x_10p['test_acc']):.3f} | Min test loss: {min(results_b0_5x_10p['test_loss']):.3f}")
print(f"Max test acc: {max(results_b2_5x_10p['test_acc']):.3f} | Min test loss: {min(results_b2_5x_10p['test_loss']):.3f}")
print(f"Max test acc: {max(results_b0_5x_20p['test_acc']):.3f} | Min test loss: {min(results_b0_5x_20p['test_loss']):.3f}")
print(f"Max test acc: {max(results_b2_5x_20p['test_acc']):.3f} | Min test loss: {min(results_b2_5x_20p['test_loss']):.3f}")
print()
print(f"Max test acc: {max(results_b0_10x_10p['test_acc']):.3f} | Min test loss: {min(results_b0_10x_10p['test_loss']):.3f}")
print(f"Max test acc: {max(results_b2_10x_10p['test_acc']):.3f} | Min test loss: {min(results_b2_10x_10p['test_loss']):.3f}")
print(f"Max test acc: {max(results_b0_10x_20p['test_acc']):.3f} | Min test loss: {min(results_b0_10x_20p['test_loss']):.3f}")
print(f"Max test acc: {max(results_b2_10x_20p['test_acc']):.3f} | Min test loss: {min(results_b2_10x_20p['test_loss']):.3f}")

In [ ]:
pretrained.plot_N_most_wrong(pred_list_b0_5x_10p, n=3)

In [ ]:
pretrained.plot_N_most_wrong(pred_list_b2_5x_10p, n=3)

In [ ]:
pretrained.plot_N_most_wrong(pred_list_b0_5x_20p, n=3)

In [ ]:
pretrained.plot_N_most_wrong(pred_list_b2_5x_20p, n=3)

In [ ]:
pretrained.plot_N_most_wrong(pred_list_b0_10x_10p, n=3)

In [ ]:
pretrained.plot_N_most_wrong(pred_list_b2_10x_10p, n=3)

In [ ]:
pretrained.plot_N_most_wrong(pred_list_b0_10x_20p, n=3)

In [ ]:
pretrained.plot_N_most_wrong(pred_list_b2_10x_20p, n=3)